In [9]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

# ==========================================
# PASO 1: GENERACIÓN DEL PANEL (6 Buckets x 6 Meses)
# ==========================================
print('--- 1. Generando datos del panel ---')
buckets = [f'B{i}' for i in range(1, 7)]
meses = [202401, 202402, 202403, 202404, 202405, 202406]

np.random.seed(42)
filas = []
for b in buckets:
    for m in meses:
        tasa_mkt = round(np.random.uniform(0.18, 0.20), 4)
        spread = round(np.random.uniform(0.02, 0.06), 4)
        tasa_ibk = round(tasa_mkt + spread, 4)
        
        conv = round(
            max(
                0.05,
                0.35 - 0.5 * tasa_ibk - 0.8 * spread + np.random.normal(0, 0.01),
            ),
            4,
        )
        
        filas.append({
            'bucket': b,
            'mes_t': m,
            'tasa_ibk': tasa_ibk,
            'tasa_mkt': tasa_mkt,
            'spread': spread,
            'conv': conv,
        })

df_panel = pd.DataFrame(filas)

# Variables logarítmicas base
df_panel['log_conv'] = np.log(df_panel['conv'])
df_panel['log_tasa_ibk'] = np.log(df_panel['tasa_ibk'])


--- 1. Generando datos del panel ---


In [10]:
df_panel.bucket.value_counts()

bucket
B1    6
B2    6
B3    6
B4    6
B5    6
B6    6
Name: count, dtype: int64

In [11]:
# ==========================================
# MÉTODO A: PANEL-OLS NATIVO (linearmodels)
# ==========================================
print('\n--- 2A. Estimando con PanelOLS Nativo ---')
df_indexed = df_panel.set_index(['bucket', 'mes_t'])

mod_panel = PanelOLS(
    dependent=df_indexed['log_conv'],
    exog=df_indexed[['log_tasa_ibk', 'spread']],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True,
)
res_panel = mod_panel.fit()
print(res_panel.summary)

beta_1_panel = res_panel.params['log_tasa_ibk']
beta_2_panel = res_panel.params['spread']



--- 2A. Estimando con PanelOLS Nativo ---
                          PanelOLS Estimation Summary                           
Dep. Variable:               log_conv   R-squared:                        0.8413
Estimator:                   PanelOLS   R-squared (Between):             -0.3451
No. Observations:                  36   R-squared (Within):               0.7403
Date:                sáb, oct 24 2026   R-squared (Overall):             -0.3430
Time:                        23:12:01   Log-likelihood                    79.380
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      60.953
Entities:                           6   P-value                           0.0000
Avg Obs:                       6.0000   Distribution:                    F(2,23)
Min Obs:                       6.0000                                           
Max Obs:                       6.0000   F-statistic (robust):     

In [12]:
print(res_panel.estimated_effects)

               estimated_effects
bucket mes_t                    
B1     202401          -1.763093
       202402          -1.781697
       202403          -1.750410
       202404          -1.775962
       202405          -1.817642
       202406          -1.799918
B2     202401          -1.824457
       202402          -1.843061
       202403          -1.811774
       202404          -1.837326
       202405          -1.879006
       202406          -1.861282
B3     202401          -1.811431
       202402          -1.830035
       202403          -1.798748
       202404          -1.824300
       202405          -1.865980
       202406          -1.848256
B4     202401          -1.779076
       202402          -1.797680
       202403          -1.766393
       202404          -1.791945
       202405          -1.833625
       202406          -1.815901
B5     202401          -1.765452
       202402          -1.784055
       202403          -1.752768
       202404          -1.778321
       202

In [13]:
# ==========================================
# MÉTODO B: DEMEANING MANUAL + OLS CLÁSICO
# ==========================================
print('\n--- 2B. Estimando con Demeaning Manual y OLS Clásico ---')

def aplicar_demeaning(df, columna):
    g_mean = df[columna].mean()
    b_mean = df.groupby('bucket')[columna].transform('mean')
    t_mean = df.groupby('mes_t')[columna].transform('mean')
    return df[columna] - b_mean - t_mean + g_mean

# Aplicamos la transformación a copia del df
df_manual = df_panel.copy()
df_manual['log_conv_dem'] = aplicar_demeaning(df_manual, 'log_conv')
df_manual['log_tasa_ibk_dem'] = aplicar_demeaning(df_manual, 'log_tasa_ibk')
df_manual['spread_dem'] = aplicar_demeaning(df_manual, 'spread')

# Regresión OLS clásica sin intercepto
X_man = df_manual[['log_tasa_ibk_dem', 'spread_dem']]
y_man = df_manual['log_conv_dem']

mod_ols = sm.OLS(y_man, X_man).fit()
print(mod_ols.summary())

beta_1_ols = mod_ols.params['log_tasa_ibk_dem']
beta_2_ols = mod_ols.params['spread_dem']


--- 2B. Estimando con Demeaning Manual y OLS Clásico ---
                                 OLS Regression Results                                
Dep. Variable:           log_conv_dem   R-squared (uncentered):                   0.841
Model:                            OLS   Adj. R-squared (uncentered):              0.832
Method:                 Least Squares   F-statistic:                              90.10
Date:                sáb, 24 oct 2026   Prob (F-statistic):                    2.58e-14
Time:                        23:12:01   Log-Likelihood:                          79.380
No. Observations:                  36   AIC:                                     -154.8
Df Residuals:                      34   BIC:                                     -151.6
Df Model:                           2                                                  
Covariance Type:            nonrobust                                                  
                       coef    std err          t      P>|t|  

In [14]:
# ==========================================
# PASO 3: VALIDACIÓN CRUZADA DE COEFICIENTES
# ==========================================
print('\n--- 3. Tabla Comparativa de Betas ---')
comparacion_betas = pd.DataFrame({
    'PanelOLS (Nativo)': [beta_1_panel, beta_2_panel],
    'Demeaning Manual + OLS': [beta_1_ols, beta_2_ols]
}, index=['log_tasa_ibk', 'spread'])
print(comparacion_betas)



--- 3. Tabla Comparativa de Betas ---
              PanelOLS (Nativo)  Demeaning Manual + OLS
log_tasa_ibk          -0.301506               -0.301506
spread                -5.588893               -5.588893


In [15]:
# ==========================================
# PASO 4: SIMULACIÓN DE NEGOCIO A FUTURO
# ==========================================
print('\n--- 4. Ejecutando simulación de negocio (Usando PanelOLS) ---')

df_simulacion = df_panel[df_panel['mes_t'] == 202406].copy()

# Incremento de +2% en tasa IBK
df_simulacion['tasa_ibk_futura'] = df_simulacion['tasa_ibk'] + 0.02
df_simulacion['spread_futuro'] = df_simulacion['tasa_ibk_futura'] - df_simulacion['tasa_mkt']

delta_log_tasa = np.log(df_simulacion['tasa_ibk_futura']) - np.log(df_simulacion['tasa_ibk'])
delta_spread = df_simulacion['spread_futuro'] - df_simulacion['spread']

# Proyección usando los coeficientes del PanelOLS
df_simulacion['log_conv_futura'] = df_simulacion['log_conv'] + (beta_1_panel * delta_log_tasa) + (beta_2_panel * delta_spread)
df_simulacion['conv_futura'] = np.exp(df_simulacion['log_conv_futura'])

monto_oferta_por_bucket = 500_000
df_simulacion['desembolso_actual'] = monto_oferta_por_bucket * df_simulacion['conv']
df_simulacion['desembolso_futuro'] = monto_oferta_por_bucket * df_simulacion['conv_futura']

total_actual = df_simulacion['desembolso_actual'].sum()
total_futuro = df_simulacion['desembolso_futuro'].sum()
impacto_colocacion = total_futuro - total_actual

print(f'Desembolso Total Base (Actual): ${total_actual:,.2f}')
print(f'Desembolso Total Simulado (Subida de tasa): ${total_futuro:,.2f}')
print(f'Impacto Neto en Colocaciones: ${impacto_colocacion:,.2f}')


--- 4. Ejecutando simulación de negocio (Usando PanelOLS) ---
Desembolso Total Base (Actual): $622,600.00
Desembolso Total Simulado (Subida de tasa): $542,565.78
Impacto Neto en Colocaciones: $-80,034.22


In [16]:
import statsmodels.formula.api as smf

# Modelo OLS con dummies explícitas para buckets y meses
mod_lsdv = smf.ols(
    'log_conv ~ log_tasa_ibk + spread + C(bucket) + C(mes_t)',
    data=df_panel,
)
res_lsdv = mod_lsdv.fit()

# Aquí verás la tabla completa con el beta, el spread y CADA una de las indicatrices
print(res_lsdv.summary())

                            OLS Regression Results                            
Dep. Variable:               log_conv   R-squared:                       0.861
Model:                            OLS   Adj. R-squared:                  0.789
Method:                 Least Squares   F-statistic:                     11.91
Date:                sáb, 24 oct 2026   Prob (F-statistic):           3.58e-07
Time:                        23:12:01   Log-Likelihood:                 79.380
No. Observations:                  36   AIC:                            -132.8
Df Residuals:                      23   BIC:                            -112.2
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -1.7631      0